## परिचय 

हा धडा खालील गोष्टींचा आढावा घेईल: 
- फंक्शन कॉलिंग म्हणजे काय आणि त्याचे उपयोग कोणते आहेत 
- OpenAI चा वापर करून फंक्शन कॉल कसे तयार करायचे 
- फंक्शन कॉल ऍप्लिकेशनमध्ये कसे समाकलित करायचे 

## शिक्षण उद्दिष्टे 

हा धडा पूर्ण केल्यावर तुम्हाला खालील गोष्टी समजतील आणि करता येतील: 

- फंक्शन कॉलिंग वापरण्याचा उद्देश 
- OpenAI सेवा वापरून फंक्शन कॉल सेटअप करणे 
- तुमच्या ऍप्लिकेशनच्या वापरासाठी प्रभावी फंक्शन कॉल डिझाइन करणे 


## फंक्शन कॉल समजून घेणे 

या धड्यासाठी, आम्हाला आमच्या शिक्षण स्टार्टअपसाठी अशी सुविधा तयार करायची आहे ज्यामुळे वापरकर्ते तांत्रिक कोर्स शोधण्यासाठी चॅटबॉट वापरू शकतील. आम्ही त्यांच्य skill level, सध्याच्या भूमिकेनुसार आणि आवडत्या तंत्रज्ञानानुसार कोर्सची शिफारस करू. 

हे पूर्ण करण्यासाठी आम्ही खालील संयोजन वापरणार आहोत: 
 - वापरकर्त्यासाठी चॅट अनुभव तयार करण्यासाठी `OpenAI`
 - वापरकर्त्याच्या विनंतीच्या आधारावर कोर्स शोधण्यासाठी `Microsoft Learn Catalog API`
 - वापरकर्त्याच्या क्वेरीला फंक्शनकडे पाठवून API विनंती करण्यासाठी `Function Calling` 

सुरुवात करण्यासाठी, आपण प्रथम का फंक्शन कॉलिंग वापरायचे आहे ते पाहूया: 


### फंक्शन कॉलिंग का 

जर तुम्ही या कोर्समधील इतर कोणतेही धडा पूर्ण केला असेल, तर तुम्हाला मोठ्या भाषा मॉडेल्स (LLMs) वापरण्याची शक्ती समजली असेल. आशा आहे की तुम्हाला त्यांच्या काही मर्यादा देखील दिसल्या असतील.

फंक्शन कॉलिंग ही OpenAI सेवेची एक वैशिष्ट्य आहे जी खालील आव्हानांचा सामना करण्यासाठी तयार केली गेली आहे:

असुसंगत प्रतिसाद स्वरूप:
- फंक्शन कॉलिंगपूर्वी, मोठ्या भाषा मॉडेलकडून प्रतिसाद अनियोजित आणि असुसंगत होते. विकसितकर्त्यांना आउटपुटमधील प्रत्येक विविधतेसाठी क्लिष्ट सत्यापन कोड लिहावा लागायचा.

बाह्य डेटासह मर्यादित समाकलन:
- या वैशिष्ट्याआधी, चॅट संदर्भात वापरासाठी अन्य भागातील डेटाचा समावेश करणे कठीण होते.

प्रतिसाद स्वरूपांना मानकीकृत करून आणि बाह्य डेटासह सुलभ समाकलन सक्षम करून, फंक्शन कॉलिंग विकास सुलभ करतो आणि अतिरिक्त सत्यापन लॉजिकची गरज कमी करतो.

वापरकर्ते "स्टॉकहोममधील सध्याचा हवामान काय आहे?" सारखे उत्तर मिळवू शकत नव्हते. कारण मॉडेल्स फक्त त्यांनी प्रशिक्षण घेतलेल्या डेटाच्या वेळेपुरते मर्यादित होते.

खालील उदाहरण पाहूया जे या समस्येचे स्पष्टीकरण करते:

समजा आपल्याला विद्यार्थ्यांच्या डेटाचा एक डेटाबेस तयार करायचा आहे जेणेकरून आपण त्यांना योग्य कोर्स सुचवू शकू. खाली आम्ही दोन विद्यार्थ्यांचे वर्णन दिले आहे जे डेटामध्ये खूप समान आहेत.


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


आम्हाला ही माहिती LLM कडे पाठवायची आहे जेणेकरून ते डेटा पार्स करू शकेल. याचा नंतर आमच्या अॅप्लिकेशनमध्ये वापर करून ते API कडे पाठवणे किंवा डेटाबेसमध्ये साठवणे शक्य होईल. 

चला दोन सारखे प्रॉम्प्ट तयार करू या ज्यात आम्ही LLM ला आम्हाला कोणती माहिती हवी आहे ते सांगू: 


आम्हाला हे आमच्या उत्पादनासाठी महत्त्वाच्या भागांचे विश्लेषण करण्यासाठी LLM ला पाठवायचे आहे. त्यामुळे आम्ही LLM ला सूचित करण्यासाठी दोन समान प्रॉम्प्ट तयार करू शकतो: 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


या दोन प्रॉम्प्ट्स तयार केल्यानंतर, आम्ही त्यांना LLM कडे `client.responses.create` वापरून पाठवू. आम्ही प्रॉम्प्ट `input` व्हेरिएबलमध्ये साठवतो आणि भूमिका `user` म्हणून देतो. हे वापरकर्त्याचा संदेश चॅटबॉटकडे पाठवताना तयार होणाऱ्या संदेशाची नक्कल करण्यासाठी आहे.



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI()

deployment="gpt-5-mini"


: 

आता आपण दोन्ही विनंत्या LLM कडे पाठवू शकतो आणि आपल्याला मिळणाऱ्या प्रतिसादाचे परीक्षण करू शकतो.


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



जरी प्रॉम्प्ट एकसारखे असले आणि वर्णने सारखी असली तरी, आपण `Grades` गुणधर्माचे वेगवेगळे स्वरूप मिळवू शकतो.

जर तुम्ही वरिल सेल अनेकदा चालवला तर, स्वरूप `3.7` किंवा `3.7 GPA` असू शकते.

कारण LLM अप्रतिबंधित डेटा लिहिलेल्या प्रॉम्प्टच्या स्वरूपात घेते आणि अप्रतिबंधित डेटा देखील परत करते. आपल्याला संरचित स्वरूप पाहिजे जेणेकरून ज्या वेळी आपण हा डेटा संग्रहित करतो किंवा वापरतो तेव्हा काय अपेक्षित आहे हे समजते.

फंक्शनल कॉलिंग वापरून, आपण निश्चित करू शकतो की आपल्याला संरचित डेटा परत मिळतो. फंक्शन कॉलिंग वापरताना, LLM प्रत्यक्षात कोणतेही फंक्शन कॉल किंवा चालवत नाही. त्याऐवजी, आपण LLM साठी त्याच्या प्रतिसादांसाठी एक संरचना तयार करतो. मग आपण त्या संरचित प्रतिसादांचा वापर करून आमच्या अॅप्लिकेशन्समध्ये कोणत्या फंक्शनला चालवायचे ते ठरवतो.
 


![फंक्शन कॉलिंग फ्लो डायग्राम](../../../../translated_images/mr/Function-Flow.083875364af4f4bb.webp)


त्यानंतर आपण फंक्शनकडून परत आलेले घेऊन ते LLM कडे परत पाठवू शकतो. नंतर LLM वापरकर्त्याच्या प्रश्नाचे उत्तर देण्यासाठी नैसर्गिक भाषेचा वापर करून प्रतिसाद देईल.


### फंक्शन कॉल वापरण्याच्या वापर प्रकरणे  

**बाह्य साधने कॉल करणे**  
चॅटबॉट वापरकर्त्यांच्या प्रश्नांची उत्तरे देण्यासाठी छान आहेत. फंक्शन कॉलिंग वापरून, चॅटबॉट वापरकर्त्यांच्या संदेशांचा उपयोग करून विशिष्ट कार्ये पूर्ण करू शकतात. उदाहरणार्थ, एखादा विद्यार्थी चॅटबॉटला विचारू शकतो, "माझ्या विषयाबाबत अधिक मदत हवी आहे, माझ्या शिक्षकाला ईमेल पाठवा". हे `send_email(to: string, body: string)` फंक्शन कॉल करू शकते.  


**API किंवा डेटाबेस क्वेरी तयार करणे**  
वापरकर्ते नैसर्गिक भाषेचा वापर करून माहिती शोधू शकतात जी नंतर एका फॉरमॅटेड क्वेरी किंवा API विनंतीत रूपांतरित केली जाते. याचे उदाहरण म्हणजे शिक्षक विचारतात, "जे विद्यार्थी शेवटचे असाइनमेंट पूर्ण केले आहेत ते कोण आहेत?" ज्यासाठी `get_completed(student_name: string, assignment: int, current_status: string)` नावाचे फंक्शन कॉल होऊ शकते.  


**रचनेकडील डेटा तयार करणे**  
वापरकर्ता मजकूराचा ब्लॉक किंवा CSV घेतो आणि त्यातून महत्त्वाची माहिती काढण्यासाठी LLM वापरू शकतो. उदाहरणार्थ, विद्यार्थी शांतता करारांबाबत विकिपीडिया लेख बदलून AI फ्लॅश कार्ड तयार करू शकतो. हे `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` नावाच्या फंक्शनचा वापर करून केले जाऊ शकते.  


## 2. आपला पहिले फंक्शन कॉल तयार करणे 

फंक्शन कॉल तयार करण्याची प्रक्रिया 3 मुख्य टप्प्यांचा समावेश करते: 
1. आपल्या फंक्शन्सची यादी आणि वापरकर्त्याचा संदेश वापरून Chat Completions API कॉल करणे 
2. क्रिया पार पाडण्यासाठी मॉडेलच्या प्रतिसादाचे वाचन करणे म्हणजे फंक्शन किंवा API कॉल चालवणे 
3. आपल्या फंक्शनमधून मिळालेल्या प्रतिसादासह पुन्हा Chat Completions API कॉल करणे जेणेकरून तो माहिती वापरून वापरकर्त्यास प्रतिसाद तयार करता येईल. 


![फंक्शन कॉलचा प्रवाह](../../../../translated_images/mr/LLM-Flow.3285ed8caf4796d7.webp)


### फंक्शन कॉलचे घटक 

#### वापरकर्त्याचा इनपुट 

पहिला टप्पा म्हणजे वापरकर्त्याचा संदेश तयार करणे. हा डाइनॅमिकली पाठ्य इनपुटची किंमत घेऊन दिला जाऊ शकतो किंवा तुम्ही येथे एक मूल्य नियुक्त करू शकता. जर तुम्ही Chat Completions API सोबत पहिल्यांदा काम करत असाल, तर आम्हाला संदेशाचा `role` आणि `content` परिभाषित करावा लागेल. 

`role` हा `system` (नियम तयार करणारा), `assistant` (मॉडेल) किंवा `user` (अंतिम वापरकर्ता) असू शकतो. फंक्शन कॉलिंगसाठी, आपण याला `user` म्हणून आणि एक उदाहरण प्रश्न म्हणून नियुक्त करू. 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### फंक्शन्स तयार करणे.

पुढे आपण एक फंक्शन आणि त्या फंक्शनच्या पॅरामीटर्सची व्याख्या करू. आपण येथे फक्त एक फंक्शन वापरणार आहोत ज्याला `search_courses` असे म्हणतात परंतु आपण अनेक फंक्शन्स तयार करू शकता.

**महत्त्वाचे** : फंक्शन्स सिस्टम मेसेजमध्ये LLM कडे समाविष्ट केले जातात आणि आपण वापरू शकणाऱ्या टोकनच्या उपलब्ध संख्येत यांचा समावेश होतो.


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**परिभाषा** 

फंक्शन डिफिनिशन संरचनेमध्ये अनेक स्तर असतात, प्रत्येकाचे स्वतःचे गुणधर्म असतात. येथे निहित संरचनेचे विघटन दिले आहे:

**टॉप लेव्हल फंक्शन गुणधर्म:**

`name` -  फंक्शनचे नाव ज्याला आपण कॉल करू इच्छिता. 

`description` - हे फंक्शन कसे काम करते याचे वर्णन. येथे विशिष्ट आणि स्पष्ट असणे महत्त्वाचे आहे 

`parameters` - मूल्ये आणि स्वरूपांची यादी जी आपण मॉडेलकडून त्याच्या प्रतिक्रियेमध्ये अपेक्षित करता 

**पॅरामीटर्स ऑब्जेक्टचे गुणधर्म:**

`type` -  पॅरामीटर्स ऑब्जेक्टचा डेटा प्रकार (साधारणपणे "object")

`properties` - त्या विशिष्ट मूल्यांची यादी जी मॉडेल आपल्या प्रतिक्रियेत वापरेल 

**वैयक्तिक पॅरामीटर गुणधर्म:**

`name` - प्रॉपर्टी कीने अप्रत्यक्षपणे निर्दिष्ट केलेले (उदा. "role", "product", "level")

`type` - या विशिष्ट पॅरामीटरचा डेटा प्रकार (उदा. "string", "number", "boolean") 

`description` - त्या विशिष्ट पॅरामीटरचे वर्णन 

**ऐच्छिक गुणधर्म:**

`required` - फंक्शन कॉल पूर्ण करण्यासाठी कोणते पॅरामीटर्स आवश्यक आहेत याची यादी असलेले अ‍ॅरे 


### फंक्शन कॉल करणे 
फंक्शन परिभाषित केल्यावर, आता आपल्याला ते Chat Completion API कॉलमध्ये समाविष्ट करावे लागेल. हे आपण विनंतीमध्ये `functions` जोडून करतो. या प्रकरणात `functions=functions` असे असते. 

`function_call` ला `auto` सेट करण्याचा एक पर्याय देखील आहे. याचा अर्थ आपण LLM ला वापरकर्त्याच्या संदेशावर आधारित कोणते फंक्शन कॉल करावे हे ठरवू देतो, त्याऐवजी आपल्याला स्वतःहून ते निश्चित करण्याची गरज नाही.


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


आता आपण प्रतिसादाकडे पाहूया आणि तो कसा फॉर्मॅट केलेला आहे ते पाहूया: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

तुम्ही पाहू शकता की फंक्शनचे नाव कॉल केल्याचे आणि वापरकर्त्याच्या संदेशातून LLM ला फंक्शनच्या आर्ग्युमेंट्ससाठी डेटा सापडला आहे. 


## 3.अ‍ॅप्लिकेशनमध्ये फंक्शन कॉल्स एकत्रित करणे.


जेव्हा आपण LLM कडून फॉरमॅट केलेला प्रतिसाद तपासला आहे, तेव्हा आता आपण याला अ‍ॅप्लिकेशनमध्ये समाकलित करू शकतो.

### फ्लो व्यवस्थापन

याला आपल्या अ‍ॅप्लिकेशनमध्ये समाकलित करण्यासाठी, आपण पुढील पावले उचलूया:

प्रथम, OpenAI सेवा कॉल करूया आणि संदेश `response_message` नावाच्या चलमध्ये संग्रहित करूया.


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


आता आपण फंक्शन परिभाषित करू ज्यामुळे Microsoft Learn API कॉल करून कोर्सची यादी मिळेल: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



सर्वोत्तम सराव म्हणून, आपण पाहू की मॉडेल फंक्शन कॉल करायचे इच्छिते का. त्यानंतर, उपलब्ध फंक्शन्सपैकी एक तयार करू आणि कॉल केलेल्या फंक्शनशी जुळवून घेऊ.
त्यानंतर आपण फंक्शनच्या आर्ग्युमेंट्स घेऊन त्यांना LLM मधील आर्ग्युमेंट्सशी जुळवू.

अखेरीस, आपण फंक्शन कॉल मेसेज आणि `search_courses` मेसेजने परत केलेल्या मूल्यांसह जोडू. यामुळे LLM ला सर्व माहिती मिळते जी त्याला
नैसर्गिक भाषेचा वापर करून वापरकर्त्याला प्रतिसाद देण्यासाठी आवश्यक आहे.


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


आता आम्ही LLM कडे अद्ययावत संदेश पाठवू ज्यामुळे आपल्याला API JSON स्वरूपाच्या प्रतिसादाऐवजी नैसर्गिक भाषेतील प्रतिसाद प्राप्त होईल.


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## कोड आव्हान 

उत्कृष्ट काम! OpenAI फंक्शन कॉलिंगचा तुमचा अभ्यास सुरू ठेवण्यासाठी तुम्ही तयार करू शकता: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - फंक्शनचे अधिक पॅरामीटर्स जे शिकणाऱ्यांना अधिक कोर्सेस शोधण्यात मदत करू शकतात. तुम्ही उपलब्ध API पॅरामीटर्स येथे शोधू शकता: 
 - आणखी एक फंक्शन कॉल तयार करा जे शिकणाऱ्यांकडून त्यांची मातृभाषा सारखी अधिक माहिती घेतो 
 - जेव्हा फंक्शन कॉल आणि/किंवा API कॉल कोणतेही योग्य कोर्सेस परत करत नाही तेव्हा त्रुटी हाताळणी तयार करा 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
